# 00 - Setup, data download and sanity checks

Run this notebook first. It (1) installs the Python modules, (2) downloads the competition data, (3) checks the hardware, the data layout, the bundled resources and the model chat templates, and (4) runs a short plumbing test with placeholder model outputs.

Nothing here sends data anywhere except the downloads from Hugging Face / Kaggle. The pipeline itself (notebooks 01-06) runs every model locally.

| notebook | stage | model (Hugging Face id) | needs |
|---|---|---|---|
| 01 | pass 1 - image-only transcription | `Qwen/Qwen3-VL-235B-A22B-Instruct` | 8 x 80 GB GPUs (or FP8, see README) |
| 02 | pass 2 - reference-examiner review | `Qwen/Qwen3.8-Flash-Next` | 8 x 80 GB GPUs |
| 03 | pass 3 - coherency check (text only) | `Qwen/Qwen3.8-Flash-Next` | as pass 2 |
| 04 | stage 1 - candidate codes | `Qwen/Qwen3.8-27B` | 1 x 80 GB GPU (or 2 x 48 GB) |
| 05 | stage 2 - audit of the candidates | `Qwen/Qwen3.8-27B` | as stage 1 |
| 06 | submission file | - | CPU |

## 1. Install modules
Run once, then restart the kernel. A CUDA-enabled `torch` and `vllm` must match your driver; if the install fails see the README ("Environment").

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, runner
LIMIT = 5         # SMOKE TEST: first 5 test rows only. Set LIMIT = None for the full 3,498 rows (finished rows are kept and skipped, so the full run resumes from the smoke test).
print('repo:', REPO, '| data:', common.DATA, '| work:', common.WORK)

## 2. Hardware, modules

In [ ]:
from misc_pipeline import sanity
hw = sanity.hardware_report()
print('modules importable:', sanity.modules_report())
print('\nHARDWARE OK for all five stages:', hw['ok'])

## 3. Competition data
Set `SOURCE` to `'huggingface'` (gated dataset `colearnlab/misconception-detection`: accept the conditions on the dataset page, then provide an access token via `HF_TOKEN` or the login prompt), `'kaggle'` (Kaggle command line tool with credentials and accepted competition rules) or `'skip'` if `data/competition_data` already exists.
The data may not be redistributed: `data/` is git-ignored.

In [ ]:
from misc_pipeline import download
SOURCE = 'huggingface'      # 'huggingface' | 'kaggle' | 'skip'
if (common.DATA / 'test.csv').is_file():
    print('data already present at', common.DATA, '- download skipped')
elif SOURCE == 'huggingface':
    download.from_huggingface()
elif SOURCE == 'kaggle':
    download.from_kaggle()
else:
    print('SOURCE is skip and no data found: place the files under', common.DATA)

In [ ]:
data_ok = sanity.dataset_report()
if data_ok:
    download.cleanup_downloads()      # removes the zip and the temporary copy (about 5 GB)
print('DATA OK:', data_ok)

## 4. Bundled resources
Prompts and rules are in the repository. The stage-2 knowledge base (`library.jsonl`, `cards.json`, `partners.json`) and the reference-working cache are derived from the competition data, so they are **not** in the public repository; they are delivered as private artifacts (README, "Private artifacts"). Put them in `resources/corpus/` and `resources/reference/` (or under `private_artifacts/`).

In [ ]:
res_ok = sanity.resources_report()
print("RESOURCES OK:", res_ok)

## 5. Model chat templates
Downloads only the tokenizer files (a few MB each) and checks that the thinking settings in `configs/models.json` change the rendered prompt. If a template ignores a setting, edit `chat_template_kwargs` there before running the stages.

In [ ]:
tmpl_ok = sanity.template_report()
print("TEMPLATES OK:", tmpl_ok)

## 6. Plumbing test (placeholder outputs, no GPU)
Builds the pass-1 jobs for three rows and runs them through the resumable runner with a placeholder backend.

In [ ]:
plumb_ok = sanity.plumbing_test(3)
print("PLUMBING OK:", plumb_ok)

## 7. Optional: download the model weights now
Weights are about 475 GB (pass 1), 360 GB (passes 2-3) and 56 GB (stages 1-2). vLLM downloads them on first use; set `PREFETCH = True` to fetch them here instead, one model at a time. Set `HF_HOME` to a disk with enough space before starting the kernel.

In [ ]:
PREFETCH = False
if PREFETCH:
    from huggingface_hub import snapshot_download
    for repo in ['Qwen/Qwen3.8-27B', 'Qwen/Qwen3.8-Flash-Next', 'Qwen/Qwen3-VL-235B-A22B-Instruct']:
        snapshot_download(repo_id=repo)

In [ ]:
print({'hardware': hw['ok'], 'data': data_ok, 'resources': res_ok, 'templates': tmpl_ok, 'plumbing': plumb_ok})